# 🧬 OntoCurator Agent — Quickstart Demo

**Agentic AI for Autonomous Ontology Term Curation in Pharmaceutical R&D**

This notebook demonstrates the full OntoCurator pipeline on a realistic cell therapy manufacturing dataset.

### What this notebook covers
1. Install and configure the package
2. Understand the 4-agent pipeline (Extract → Map → Resolve → Govern)
3. Run curation on 3 real-world lab report samples
4. Visualise the results: auto-curation rates, governance routing, term mappings
5. Inspect the full audit trail for any term

---
**Model:** `gpt-5.2` (OpenAI)  
**Ontologies searched:** OBI, ChEBI, GO, CL, EFO, BAO, NCIT  
**Requires:** `OPENAI_API_KEY` environment variable

## 1. Install the package

In [ ]:
# Install the package (editable install from local source)
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "..[notebooks]"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

## 2. Configure API Keys

Set your keys either in a `.env` file at the project root or directly below.

```
# .env (create this file, it is git-ignored)
OPENAI_API_KEY=sk-...
BIOPORTAL_API_KEY=...   # optional — get free key at bioportal.bioontology.org
```

In [ ]:
import os
from dotenv import load_dotenv

# Load .env from project root (one directory up from notebooks/)
load_dotenv(dotenv_path="../.env")

# ── Uncomment to set keys directly in this cell (don't commit!) ──
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["BIOPORTAL_API_KEY"] = "..."  # optional

# Verify
key = os.environ.get("OPENAI_API_KEY", "")
if key:
    print(f"✅ OPENAI_API_KEY loaded ({key[:8]}...{key[-4:]})")
else:
    print("❌ OPENAI_API_KEY not found — set it before running cells below")

bp_key = os.environ.get("BIOPORTAL_API_KEY", "")
print(f"{'✅' if bp_key else '⚠️ '} BIOPORTAL_API_KEY {'loaded' if bp_key else 
}")

## 3. Test Data — Cell Therapy Lab Reports

Three realistic lab report excerpts from a CAR-T cell manufacturing process.
Each contains multiple biomedical terms that may need ontology registration.

In [ ]:
# ── Test documents ────────────────────────────────────────────────────────────

TEST_DOCUMENTS = {
    "DOC-CAR-T-001": """
    Day 5 in-process assessment for CAR-T Batch CT-2024-045.
    Viable cell density was 3.2 × 10^6 cells/mL as determined by trypan blue exclusion
    using the Vi-CELL XR automated cell counter. Cell viability was 94.2%.
    CD3+ T cells comprised 87% of the total cell population by flow cytometry
    using a CD3/CD4/CD8 multi-colour panel.
    Transduction efficiency with the anti-CD19 CAR lentiviral vector was 72% CAR-positive
    cells as measured by anti-idiotype antibody staining.
    Glucose concentration was 8.4 mM; lactate was 12.1 mM by YSI analysis.
    """,

    "DOC-QC-RELEASE-002": """
    Final release testing for lot CT-2024-045 completed on Day 14.
    Sterility testing by direct inoculation showed no growth at 14 days.
    Mycoplasma testing by PCR (MycoSEQ assay) was negative.
    Endotoxin by Limulus Amebocyte Lysate (LAL) assay: 0.3 EU/mL (specification: ≤ 5 EU/mL).
    Vector copy number (VCN) determined by droplet digital PCR: 2.4 copies per cell.
    Potency assessment by cytotoxicity assay against Raji cells at E:T ratio 5:1:
    specific lysis 68% (specification: ≥ 20%).
    Final cell dose: 2.1 × 10^8 total CAR+ T cells.
    """,

    "DOC-FORMULATION-003": """
    Cryopreservation formulation development for CAR-T cell product.
    Cells were formulated in CryoStor CS10 cryopreservation medium containing
    10% dimethyl sulfoxide (DMSO) and 90% human serum albumin (HSA) solution.
    Controlled-rate freezing was performed using a Planer Kryo 560 controlled rate freezer
    at a cooling rate of -1°C/min to -80°C.
    Post-thaw viability was assessed by propidium iodide exclusion assay.
    Recovery rate was 89% viable cells post-thaw.
    Osmolality was measured by freezing point depression osmometry.
    """
}

for doc_id, text in TEST_DOCUMENTS.items():
    print(f"📄 {doc_id}: {len(text.split())} words")

## 4. Initialise the Pipeline

In [ ]:
from onto_curator import OntoCuratorPipeline

pipeline = OntoCuratorPipeline.from_env(
    model="gpt-5.2",                  # OpenAI GPT-5.2
    min_extraction_confidence=0.55,   # keep terms with ≥55% extraction confidence
    auto_approve_threshold=0.88,      # ≥88% mapping confidence → auto-approve
    human_review_threshold=0.60,      # 60-88% → human review
                                      # <60% → escalate
)

print(f"✅ OntoCuratorPipeline ready (model: {pipeline.model})")

## 5. Run Curation — Document 1: In-Process Assessment

In [ ]:
from rich import print as rprint
from rich.table import Table
from rich.console import Console

console = Console()

doc_id = "DOC-CAR-T-001"
print(f"\n🔄 Curating {doc_id}...\n")

summary_1 = pipeline.curate(
    text=TEST_DOCUMENTS[doc_id],
    source_document=doc_id
)

print(f"\n✅ Run {summary_1.run_id} complete")
print(f"   Total terms extracted : {summary_1.total_terms}")
print(f"   Auto-approved         : {summary_1.auto_approved}")
print(f"   Pending human review  : {summary_1.pending_human_review}")
print(f"   Escalated             : {summary_1.escalated}")
print(f"   New term proposals    : {summary_1.new_terms_proposed}")
print(f"   Conflict count        : {summary_1.conflict_count}")
print(f"   Auto-curation rate    : {summary_1.auto_curation_rate}%")

In [ ]:
# ── Pretty-print per-term results ─────────────────────────────────────────────

table = Table(
    title=f"Curation Results — {doc_id}",
    show_lines=True
)
table.add_column("Term", style="cyan", max_width=28)
table.add_column("Type", style="yellow", max_width=12)
table.add_column("Mapped To", style="green", max_width=18)
table.add_column("Conf.", justify="right", max_width=6)
table.add_column("Decision", max_width=14)
table.add_column("Steward", style="dim", max_width=28)

DECISION_STYLE = {
    "approved": "[bold green]✅ approved[/]",
    "pending_review": "[bold yellow]👁 review[/]",
    "escalated": "[bold red]🚨 escalated[/]",
    "rejected": "[bold red]❌ rejected[/]",
}

for result in summary_1.results:
    m = result.mapping
    g = result.governance
    mapped_to = m.top_match.ontology_id if m.top_match else "NEW TERM"
    steward = g.assigned_steward or "—"
    decision_text = DECISION_STYLE.get(result.status, result.status)

    table.add_row(
        m.candidate.text,
        str(m.candidate.entity_type),
        mapped_to,
        f"{m.mapping_confidence:.2f}",
        decision_text,
        steward,
    )

console.print(table)

## 6. Run All 3 Documents and Compare

In [ ]:
summaries = {}

for doc_id, text in TEST_DOCUMENTS.items():
    print(f"🔄 Curating {doc_id}...", end=" ")
    summary = pipeline.curate(text=text, source_document=doc_id)
    summaries[doc_id] = summary
    print(f"done — {summary.total_terms} terms, {summary.auto_curation_rate}% auto-curated")

In [ ]:
import pandas as pd

# Build comparison DataFrame
rows = []
for doc_id, s in summaries.items():
    rows.append({
        "Document": doc_id,
        "Total Terms": s.total_terms,
        "Auto-Approved": s.auto_approved,
        "Human Review": s.pending_human_review,
        "Escalated": s.escalated,
        "New Terms": s.new_terms_proposed,
        "Conflicts": s.conflict_count,
        "Auto-Curation Rate %": s.auto_curation_rate,
    })

df = pd.DataFrame(rows)
df.style.highlight_max(
    subset=["Auto-Curation Rate %"], color="lightgreen"
).highlight_max(
    subset=["Conflicts"], color="#ffcccc"
)

## 7. Visualise Governance Routing Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("OntoCurator — Governance Routing Summary", fontsize=14, fontweight="bold")

# ── Left: stacked bar per document ────────────────────────────────────────────
doc_labels = [d.replace("DOC-", "") for d in summaries.keys()]
auto_vals   = [s.auto_approved        for s in summaries.values()]
review_vals = [s.pending_human_review for s in summaries.values()]
escal_vals  = [s.escalated            for s in summaries.values()]

x = np.arange(len(doc_labels))
axes[0].bar(x, auto_vals,   label="Auto-Approved",  color="#4CAF50")
axes[0].bar(x, review_vals, label="Human Review",   color="#FFC107", bottom=auto_vals)
axes[0].bar(x, escal_vals,  label="Escalated",      color="#F44336",
            bottom=[a + r for a, r in zip(auto_vals, review_vals)])
axes[0].set_xticks(x)
axes[0].set_xticklabels(doc_labels, rotation=15, ha="right")
axes[0].set_ylabel("Number of Terms")
axes[0].set_title("Governance Routing by Document")
axes[0].legend()

# ── Right: overall pie ────────────────────────────────────────────────────────
total_auto   = sum(s.auto_approved        for s in summaries.values())
total_review = sum(s.pending_human_review for s in summaries.values())
total_escal  = sum(s.escalated            for s in summaries.values())

wedges, texts, autotexts = axes[1].pie(
    [total_auto, total_review, total_escal],
    labels=["Auto-Approved", "Human Review", "Escalated"],
    colors=["#4CAF50", "#FFC107", "#F44336"],
    autopct="%1.0f%%",
    startangle=90,
    textprops={"fontsize": 11},
)
axes[1].set_title("Overall Routing Distribution")

plt.tight_layout()
plt.savefig("governance_routing.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to governance_routing.png")

## 8. Inspect a Single Term — Confidence Breakdown & Audit Trail

In [ ]:
# Pick the first result from document 1 and inspect it in depth
result = summaries["DOC-CAR-T-001"].results[0]

m = result.mapping
g = result.governance

print("═" * 60)
print(f"  TERM: {m.candidate.text}")
print("═" * 60)
print(f"  Normalised text  : {m.candidate.normalized_text}")
print(f"  Entity type      : {m.candidate.entity_type}")
print(f"  Extraction conf. : {m.candidate.confidence:.2f}")
print(f"  Context          : {m.candidate.context[:120]}")
print()
print("  ── Mapping ──────────────────────────────────")
if m.top_match:
    print(f"  Top match        : {m.top_match.ontology_id} — {m.top_match.matched_label}")
    print(f"  Ontology         : {m.top_match.ontology_name}")
    print(f"  Definition       : {(m.top_match.definition or 'n/a')[:120]}")
    print(f"  Match source     : {m.top_match.match_source}")
else:
    print("  Top match        : ⚠️  No match found — new term proposed")
print(f"  Mapping conf.    : {m.mapping_confidence:.2f}")
print(f"  Mapper reasoning : {m.mapper_reasoning or 'n/a'}")
print()
print("  ── Governance ───────────────────────────────")
print(f"  Decision         : {g.action}")
print(f"  Reason           : {g.reason}")
print(f"  Priority         : {g.priority}")
print(f"  Assigned steward : {g.assigned_steward or '—'}")
print()
print("  ── Audit Trail ──────────────────────────────")
for step in result.audit_trail:
    print(f"   [{step['step']:8}] {step['agent']}")
print("═" * 60)

## 9. Build the Human Review Queue

Collect all terms that need human review or escalation across all documents.

In [ ]:
review_queue = []

for doc_id, summary in summaries.items():
    for result in summary.results:
        if result.status in ("pending_review", "escalated"):
            review_queue.append({
                "Document": doc_id,
                "Term": result.mapping.candidate.text,
                "Type": result.mapping.candidate.entity_type,
                "Status": result.status,
                "Top Match": (
                    f"{result.mapping.top_match.ontology_id} ({result.mapping.top_match.matched_label})"
                    if result.mapping.top_match else "UNMAPPED"
                ),
                "Conf.": result.mapping.mapping_confidence,
                "Steward": result.governance.assigned_steward or "—",
                "New Term?": "⚠️ Yes" if result.mapping.needs_new_term else "No",
            })

df_queue = pd.DataFrame(review_queue).sort_values(
    ["Status", "Conf."],
    ascending=[True, True]  # escalated first, then by lowest confidence
).reset_index(drop=True)

print(f"📋 Human Review Queue: {len(df_queue)} terms")
df_queue.style.applymap(
    lambda v: "background-color: #ffcccc" if v == "escalated" else
              "background-color: #fff3cd" if v == "pending_review" else "",
    subset=["Status"]
)

## 10. Confidence Score Distribution

In [ ]:
import seaborn as sns

all_confs = []
all_statuses = []

for summary in summaries.values():
    for result in summary.results:
        all_confs.append(result.mapping.mapping_confidence)
        all_statuses.append(result.status)

conf_df = pd.DataFrame({"confidence": all_confs, "status": all_statuses})

fig, ax = plt.subplots(figsize=(10, 4))

palette = {"approved": "#4CAF50", "pending_review": "#FFC107", "escalated": "#F44336"}
for status, grp in conf_df.groupby("status"):
    ax.hist(grp["confidence"], bins=15, alpha=0.65, label=status,
            color=palette.get(status, "grey"), edgecolor="white")

# Draw threshold lines
ax.axvline(0.60, color="orange", linestyle="--", linewidth=1.5, label="Human review threshold (0.60)")
ax.axvline(0.88, color="green",  linestyle="--", linewidth=1.5, label="Auto-approve threshold (0.88)")

ax.set_xlabel("Mapping Confidence", fontsize=12)
ax.set_ylabel("Number of Terms", fontsize=12)
ax.set_title("Mapping Confidence Distribution by Governance Decision", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Export Curation Results to CSV

In [ ]:
all_rows = []
for doc_id, summary in summaries.items():
    for result in summary.results:
        m = result.mapping
        all_rows.append({
            "run_id":               summary.run_id,
            "document":             doc_id,
            "term":                 m.candidate.text,
            "normalized_term":      m.candidate.normalized_text,
            "entity_type":          m.candidate.entity_type,
            "extraction_confidence":m.candidate.confidence,
            "ontology_id":          m.top_match.ontology_id if m.top_match else None,
            "ontology_label":       m.top_match.matched_label if m.top_match else None,
            "ontology_name":        m.top_match.ontology_name if m.top_match else None,
            "mapping_confidence":   m.mapping_confidence,
            "needs_new_term":       m.needs_new_term,
            "governance_action":    result.governance.action,
            "governance_reason":    result.governance.reason,
            "assigned_steward":     result.governance.assigned_steward,
            "status":               result.status,
            "mapper_reasoning":     m.mapper_reasoning,
        })

df_export = pd.DataFrame(all_rows)
df_export.to_csv("curation_results.csv", index=False)
print(f"✅ Exported {len(df_export)} rows to curation_results.csv")
df_export.head(10)

## 12. Next Steps

| Step | What to do |
|------|------------|
| **BioPortal integration** | Set `BIOPORTAL_API_KEY` for richer ontology matches (OBI, ChEBI, CL, DOID) |
| **Custom confidence thresholds** | Tune `auto_approve_threshold` and `human_review_threshold` based on your QA requirements |
| **Domain extension** | Add governance rules for a specific domain (e.g., formulation, genomics) in `governance_gate.py` |
| **Knowledge graph export** | Pipe `curation_results.csv` into Neo4j or Apache Jena |
| **Active learning** | Collect human corrections from the review queue to improve mapping accuracy over time |
| **Batch processing** | Use `pipeline.curate()` in a loop over thousands of documents |

---
*Built by Ali Shahmohammadi — [github.com/alishahmohammadi22](https://github.com/alishahmohammadi22)*